# Receipt Credit Card Agent Comparison Notebook

This notebook compares full pipeline variants on the same receipt:

1. `Version 1` = `tesseract` perception + `planning v1` classical classification
2. `Version 2` = `trocr` perception + `planning v2` transformer-based classification
3. `Labels Reference` = dataset-provided `labels` text + configurable planning version

All three share the same LLM control phase with live web research through `tool_registry`.


In [ ]:
from pathlib import Path
import base64
import html
import importlib
import json
import sys
from io import BytesIO

import pandas as pd
from PIL import Image
from IPython.display import HTML, Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.perception as perception_module
import src.planning as planning_module
import src.control as control_module
import src.utils as utils_module

importlib.reload(perception_module)
importlib.reload(planning_module)
importlib.reload(control_module)
importlib.reload(utils_module)

ReceiptPerception = perception_module.ReceiptPerception
ReceiptPlanner = planning_module.ReceiptPlanner
CreditCardRecommender = control_module.CreditCardRecommender
preferred_receipts_dir = utils_module.preferred_receipts_dir
list_receipt_images = utils_module.list_receipt_images


## Settings

In [ ]:
receipts_dir = preferred_receipts_dir()
receipt_candidates = list_receipt_images(receipts_dir)
if not receipt_candidates:
    raise FileNotFoundError(f'No receipt images found in: {receipts_dir}')

receipt_path = receipt_candidates[0]
pipeline_configs = [
    {
        'label': 'version_1_non_dl',
        'perception_method': 'tesseract',
        'planning_version': 'v1',
    },
    {
        'label': 'version_2_dl',
        'perception_method': 'trocr',
        'planning_version': 'v2',
    },
]

if receipts_dir.name == 'receipts':
    pipeline_configs.append({
        'label': 'labels_reference',
        'perception_method': 'labels',
        'planning_version': 'v2',
    })

run_control = True

receipt_path


## View Selected Receipt

In [ ]:
receipt_image = Image.open(receipt_path)
display(receipt_image)
print('Image size:', receipt_image.size)

## Initialize Pipeline

In [ ]:
perception = ReceiptPerception()
planner = ReceiptPlanner()
recommender = CreditCardRecommender() if run_control else None

## Helper Functions

In [ ]:
def run_pipeline_config(image_path, config, run_control=True):
    try:
        ocr_result = perception.extract_text(image_path, method=config['perception_method'])
        spending_profile = planner.build_spending_profile(ocr_result.text, version=config['planning_version'])
        recommendation = recommender.recommend_card(spending_profile) if run_control else None
        return {
            'label': config['label'],
            'perception_method': config['perception_method'],
            'planning_version': config['planning_version'],
            'status': 'ok',
            'error': None,
            'ocr_result': ocr_result,
            'spending_profile': spending_profile,
            'recommendation': recommendation,
        }
    except Exception as exc:
        return {
            'label': config['label'],
            'perception_method': config['perception_method'],
            'planning_version': config['planning_version'],
            'status': 'error',
            'error': str(exc),
            'ocr_result': None,
            'spending_profile': None,
            'recommendation': None,
        }


def summarize_result(result):
    if result['status'] != 'ok':
        return {
            'label': result['label'],
            'perception_method': result['perception_method'],
            'planning_version': result['planning_version'],
            'status': 'error',
            'error': result['error'],
            'merchant': None,
            'total_amount': None,
            'line_item_count': None,
            'non_zero_categories': None,
            'primary_card': None,
            'primary_issuer': None,
            'estimated_value': None,
        }

    profile = result['spending_profile']
    recommendation = result['recommendation']
    non_zero_categories = [
        f"{key}: {value:.2f}"
        for key, value in profile.category_totals.items()
        if value > 0
    ]
    return {
        'label': result['label'],
        'perception_method': result['perception_method'],
        'planning_version': result['planning_version'],
        'status': 'ok',
        'error': None,
        'merchant': profile.merchant,
        'total_amount': profile.total_amount,
        'line_item_count': len(profile.line_items),
        'non_zero_categories': ', '.join(non_zero_categories) if non_zero_categories else '[none]',
        'primary_card': recommendation.primary_card if recommendation else '[skipped]',
        'primary_issuer': recommendation.primary_issuer if recommendation else '[skipped]',
        'estimated_value': recommendation.primary_estimated_value if recommendation else None,
    }


def thumbnail_html(image_path, width=220, max_height=700):
    image = Image.open(image_path).convert('RGB')
    thumbnail = image.copy()
    thumbnail.thumbnail((width, max_height))
    buffer = BytesIO()
    thumbnail.save(buffer, format='JPEG')
    encoded = base64.b64encode(buffer.getvalue()).decode('utf-8')
    return (
        f'<img src="data:image/jpeg;base64,{encoded}" ' 
        f'style="width:{thumbnail.width}px; border:1px solid #ddd; border-radius:6px;" />'
    )


def parsed_text_html(text, width=420, max_height=320):
    safe_text = html.escape(text or '')
    return (
        f'<div style="width:{width}px; max-height:{max_height}px; overflow:auto; ' 
        'white-space:pre-wrap; font-family:monospace; font-size:12px; line-height:1.35; ' 
        'border:1px solid #ddd; border-radius:6px; padding:8px; background:#fafafa;">'
        f'{safe_text}</div>'
    )


## Run Pipeline Variants on One Receipt


In [ ]:
results = [run_pipeline_config(receipt_path, config, run_control=run_control) for config in pipeline_configs]
summary_df = pd.DataFrame([summarize_result(result) for result in results])
display(summary_df)


## OCR Text Previews

In [ ]:
for result in results:
    heading = f"### {result['label']} ({result['perception_method']} + {result['planning_version']})"
    display(Markdown(heading))
    if result['status'] != 'ok':
        print('ERROR:', result['error'])
        print()
        continue

    preview = result['ocr_result'].text[:2000] if result['ocr_result'].text else '[No text extracted]'
    print(preview)
    print()


## Spending Profile Comparison

In [ ]:
profiles_df = pd.DataFrame([
    {
        'label': result['label'],
        'perception_method': result['perception_method'],
        'planning_version': result['planning_version'],
        'status': result['status'],
        'merchant': result['spending_profile'].merchant if result['status'] == 'ok' else None,
        'groceries': result['spending_profile'].category_totals['groceries'] if result['status'] == 'ok' else None,
        'dining': result['spending_profile'].category_totals['dining'] if result['status'] == 'ok' else None,
        'travel': result['spending_profile'].category_totals['travel'] if result['status'] == 'ok' else None,
        'gas': result['spending_profile'].category_totals['gas'] if result['status'] == 'ok' else None,
        'entertainment': result['spending_profile'].category_totals['entertainment'] if result['status'] == 'ok' else None,
        'shopping': result['spending_profile'].category_totals['shopping'] if result['status'] == 'ok' else None,
        'healthcare': result['spending_profile'].category_totals['healthcare'] if result['status'] == 'ok' else None,
        'other': result['spending_profile'].category_totals['other'] if result['status'] == 'ok' else None,
        'total_amount': result['spending_profile'].total_amount if result['status'] == 'ok' else None,
        'line_item_count': len(result['spending_profile'].line_items) if result['status'] == 'ok' else None,
    }
    for result in results
])
display(profiles_df)


## Recommendation Comparison

In [ ]:
if run_control:
    recommendations_df = pd.DataFrame([
        {
            'label': result['label'],
            'perception_method': result['perception_method'],
            'planning_version': result['planning_version'],
            'status': result['status'],
            'primary_card': result['recommendation'].primary_card if result['status'] == 'ok' and result['recommendation'] else None,
            'primary_issuer': result['recommendation'].primary_issuer if result['status'] == 'ok' and result['recommendation'] else None,
            'primary_value': result['recommendation'].primary_estimated_value if result['status'] == 'ok' and result['recommendation'] else None,
            'runner_up_card': result['recommendation'].runner_up_card if result['status'] == 'ok' and result['recommendation'] else None,
            'runner_up_issuer': result['recommendation'].runner_up_issuer if result['status'] == 'ok' and result['recommendation'] else None,
            'runner_up_value': result['recommendation'].runner_up_estimated_value if result['status'] == 'ok' and result['recommendation'] else None,
        }
        for result in results
    ])
    display(recommendations_df)
else:
    print('Control step skipped.')


## Full Structured Output for One Method

In [ ]:
available_labels = [result['label'] for result in results]
selected_pipeline_label = 'labels_reference' if 'labels_reference' in available_labels else available_labels[0]
selected_result = next(result for result in results if result['label'] == selected_pipeline_label)

if selected_result['status'] != 'ok':
    print('ERROR:', selected_result['error'])
else:
    combined_output = {
        'pipeline': {
            'label': selected_result['label'],
            'perception_method': selected_result['perception_method'],
            'planning_version': selected_result['planning_version'],
        },
        'ocr': {
            'method': selected_result['ocr_result'].method,
            'image_path': selected_result['ocr_result'].image_path,
            'confidence': selected_result['ocr_result'].confidence,
            'metadata': selected_result['ocr_result'].metadata,
            'text': selected_result['ocr_result'].text,
        },
        'planning': selected_result['spending_profile'].as_dict(),
        'control': selected_result['recommendation'].as_dict() if selected_result['recommendation'] else None,
    }
    print(json.dumps(combined_output, indent=2))


## Batch Comparison Across a Few Receipts

In [ ]:
sample_receipts = list_receipt_images(preferred_receipts_dir())[:3]
batch_rows = []

for image_path in sample_receipts:
    for config in pipeline_configs:
        result = run_pipeline_config(image_path, config, run_control=run_control)
        batch_rows.append({
            "image": thumbnail_html(image_path),
            "receipt": image_path.name,
            "label": result["label"],
            "perception_method": result["perception_method"],
            "planning_version": result["planning_version"],
            "parsed_text": parsed_text_html(result["ocr_result"].text) if result["status"] == "ok" else None,
            "status": result["status"],
            "merchant": result["spending_profile"].merchant if result["status"] == "ok" else None,
            "total_amount": result["spending_profile"].total_amount if result["status"] == "ok" else None,
            "line_item_count": len(result["spending_profile"].line_items) if result["status"] == "ok" else None,
            "primary_card": result["recommendation"].primary_card if result["status"] == "ok" and result["recommendation"] else None,
            "estimated_value": result["recommendation"].primary_estimated_value if result["status"] == "ok" and result["recommendation"] else None,
            "error": result["error"],
        })

batch_df = pd.DataFrame(batch_rows, columns=[
    "image",
    "receipt",
    "label",
    "perception_method",
    "planning_version",
    "parsed_text",
    "status",
    "merchant",
    "total_amount",
    "line_item_count",
    "primary_card",
    "estimated_value",
    "error",
])

table_html = batch_df.to_html(escape=False, index=False)
table_html = table_html.replace(
    '<table border="1" class="dataframe">',
    '<table border="1" class="dataframe comparison-table">'
)

display(HTML('''
<style>
.comparison-table td, .comparison-table th {
    vertical-align: top;
    text-align: left;
    padding: 8px;
}
</style>
''' + table_html))
